# 🏎️ F1 Predictions: 2026 Japanese Grand Prix (Round 3)## Suzuka Circuit — March 27-29, 2026**Model version:** v0.1 (ELO-based with track adjustments)  **Data sources:** FastF1 API (2023-2026), OpenF1  **Last updated:** March 24, 2026

In [ ]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

fastf1.Cache.enable_cache('../data/cache')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

print("Setup complete ✓")


## 1. Data CollectionPull race results from:- **2023-2025 Japanese GP** (Suzuka-specific performance)- **Late 2025 season** (last 5 races for recent form)- **2026 season** (Australia + China results)

In [ ]:
# Collect all relevant race results into a unified DataFrame
all_results = []

# Helper to extract results
def get_race_results(year, round_num, race_name):
    session = fastf1.get_session(year, round_num, 'R')
    session.load(telemetry=False, weather=False, messages=False)
    df = session.results[['DriverNumber', 'FullName', 'TeamName', 'Position', 'GridPosition', 'Points', 'Status']].copy()
    df['Year'] = year
    df['Round'] = round_num
    df['RaceName'] = race_name
    df['IsJapan'] = 'Japan' in race_name or 'Suzuka' in race_name
    return df

# Japanese GPs 2023-2025
schedule_2023 = fastf1.get_event_schedule(2023)
japan_2023 = schedule_2023[schedule_2023['EventName'].str.contains('Japan')].iloc[0]
all_results.append(get_race_results(2023, int(japan_2023['RoundNumber']), 'Japanese Grand Prix'))

schedule_2024 = fastf1.get_event_schedule(2024)
japan_2024 = schedule_2024[schedule_2024['EventName'].str.contains('Japan')].iloc[0]
all_results.append(get_race_results(2024, int(japan_2024['RoundNumber']), 'Japanese Grand Prix'))

schedule_2025 = fastf1.get_event_schedule(2025)
japan_2025 = schedule_2025[schedule_2025['EventName'].str.contains('Japan')].iloc[0]
all_results.append(get_race_results(2025, int(japan_2025['RoundNumber']), 'Japanese Grand Prix'))

# Last 5 races of 2025
races_2025 = schedule_2025[schedule_2025['RoundNumber'] > 0]
for _, race in races_2025.tail(5).iterrows():
    rnd = int(race['RoundNumber'])
    all_results.append(get_race_results(2025, rnd, race['EventName']))

# 2026 races (Australia + China)
all_results.append(get_race_results(2026, 1, 'Australian Grand Prix'))
all_results.append(get_race_results(2026, 2, 'Chinese Grand Prix'))

# Combine
df = pd.concat(all_results, ignore_index=True)
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')
df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')

print(f"Collected {len(df)} driver-race entries across {df[['Year','Round']].drop_duplicates().shape[0]} races")
print(f"Unique drivers: {df['FullName'].nunique()}")
print(f"\nRaces included:")
for _, row in df[['Year','RaceName']].drop_duplicates().iterrows():
    print(f"  {row['Year']} {row['RaceName']}")


## 2. Driver Ratings (ELO System)Build an ELO-style rating for each driver based on recent results. Drivers gain/lose rating based on finishing position relative to expected performance.**Key features:**- Recent races weighted more heavily (exponential decay)- Grid position vs finish position captures race pace vs qualifying pace- Track-specific bonus for Suzuka performance

In [ ]:
# Build driver ELO ratings
# Start everyone at 1500, adjust based on race results

elo_ratings = {}
K = 32  # ELO K-factor

def get_elo(driver):
    if driver not in elo_ratings:
        elo_ratings[driver] = 1500.0
    return elo_ratings[driver]

def update_elo(results_df):
    """Update ELO based on race results. Higher finish = more points gained."""
    drivers = results_df.sort_values('Position')
    n = len(drivers)
    
    for _, row in drivers.iterrows():
        driver = row['FullName']
        pos = row['Position']
        if pd.isna(pos) or pos > 20:
            continue
        
        current_elo = get_elo(driver)
        
        # Expected position based on ELO (lower ELO rank = expected worse finish)
        all_elos = [(get_elo(r['FullName']), r['FullName']) for _, r in drivers.iterrows() if not pd.isna(r['Position'])]
        all_elos.sort(reverse=True)
        expected_pos = next((i+1 for i, (e, name) in enumerate(all_elos) if name == driver), n)
        
        # Points gained/lost based on beating/losing to expectation
        position_delta = expected_pos - pos  # positive = beat expectations
        elo_change = K * position_delta / n
        
        elo_ratings[driver] = current_elo + elo_change

# Process races in chronological order
races_ordered = df.sort_values(['Year', 'Round']).groupby(['Year', 'Round'])

for (year, rnd), race_df in races_ordered:
    update_elo(race_df)

# Current ratings
elo_df = pd.DataFrame([
    {'Driver': driver, 'ELO': rating} 
    for driver, rating in elo_ratings.items()
]).sort_values('ELO', ascending=False)

# Filter to 2026 grid only
grid_2026 = df[df['Year'] == 2026]['FullName'].unique()
current_elo = elo_df[elo_df['Driver'].isin(grid_2026)].reset_index(drop=True)

print("=== Current Driver ELO Ratings (2026 Grid) ===\n")
for i, row in current_elo.iterrows():
    bar = '█' * int((row['ELO'] - 1400) / 5)
    print(f"  {row['Driver']:25s}  {row['ELO']:7.1f}  {bar}")


## 3. Suzuka Track AdjustmentSome drivers consistently perform better at specific tracks. We calculate a "Suzuka factor" based on historical results at the Japanese GP.

In [ ]:
# Suzuka-specific performance (2023-2025)
suzuka_results = df[df['IsJapan'] == True].copy()
suzuka_avg = suzuka_results.groupby('FullName').agg(
    avg_position=('Position', 'mean'),
    avg_grid=('GridPosition', 'mean'),
    races=('Position', 'count'),
    best_finish=('Position', 'min')
).reset_index()

# Suzuka factor: how much better/worse than average grid position
suzuka_avg['suzuka_factor'] = suzuka_avg['avg_grid'] - suzuka_avg['avg_position']  # positive = gains positions

# Map to 2026 drivers (some may not have Suzuka history)
suzuka_factors = {}
for _, row in suzuka_avg.iterrows():
    suzuka_factors[row['FullName']] = row['suzuka_factor']

print("=== Suzuka Track Performance (2023-2025) ===\n")
print(f"{'Driver':25s}  {'Avg Finish':>10s}  {'Avg Grid':>9s}  {'Gain/Loss':>9s}  {'Races':>5s}")
print("-" * 65)
for _, row in suzuka_avg.sort_values('avg_position').iterrows():
    gain = f"+{row['suzuka_factor']:.1f}" if row['suzuka_factor'] > 0 else f"{row['suzuka_factor']:.1f}"
    print(f"  {row['FullName']:25s}  P{row['avg_position']:>7.1f}  P{row['avg_grid']:>6.1f}  {gain:>9s}  {int(row['races']):>5d}")


## 4. Grid-to-Race Position ModelA key predictor: how well does a driver convert their grid position into race results? This linear regression model captures the relationship between qualifying and race finish.

In [ ]:
# Train a model: Grid Position → Race Finish Position
# Use all collected data
model_data = df.dropna(subset=['Position', 'GridPosition']).copy()
model_data = model_data[model_data['Position'] <= 20]  # Only classified finishers

X = model_data[['GridPosition']].values
y = model_data['Position'].values

model = LinearRegression()
model.fit(X, y)

print(f"Grid-to-Race Model: Finish = {model.coef_[0]:.3f} × Grid + {model.intercept_:.3f}")
print(f"R² score: {model.score(X, y):.3f}")
print(f"\nInterpretation: Starting P1 predicts finishing ~P{model.predict([[1]])[0]:.1f}")
print(f"               Starting P10 predicts finishing ~P{model.predict([[10]])[0]:.1f}")
print(f"               Starting P20 predicts finishing ~P{model.predict([[20]])[0]:.1f}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(model_data['GridPosition'], model_data['Position'], alpha=0.3, color='#ff006e', s=30)
grid_range = np.linspace(1, 22, 100).reshape(-1, 1)
ax.plot(grid_range, model.predict(grid_range), color='#06ffa5', linewidth=2, label='Trend')
ax.plot([1, 22], [1, 22], '--', color='#3a86ff', alpha=0.5, label='No change')
ax.set_xlabel('Grid Position', fontsize=12)
ax.set_ylabel('Race Finish Position', fontsize=12)
ax.set_title('Grid Position vs Race Finish (2023-2026)', fontsize=14)
ax.legend()
ax.set_xlim(0.5, 22.5)
ax.set_ylim(0.5, 22.5)
ax.invert_yaxis()
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('../data/grid_vs_race.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved to data/grid_vs_race.png")


## 5. Japan GP PredictionsCombining:1. **ELO rating** (overall driver strength)2. **Suzuka factor** (track-specific adjustment)  3. **Grid-to-race model** (qualifying → race conversion)4. **2026 form** (weighted recent results)

In [ ]:
# Build prediction for Japan GP
# Since qualifying hasn't happened yet, we estimate grid from ELO + Suzuka factor

predictions = []
for driver in grid_2026:
    elo = get_elo(driver)
    suzuka = suzuka_factors.get(driver, 0)  # 0 if no Suzuka history
    
    # 2026 form: average position in 2026 races
    form_2026 = df[(df['Year'] == 2026) & (df['FullName'] == driver)]['Position'].mean()
    if pd.isna(form_2026):
        form_2026 = 15  # default for unknowns
    
    # Composite score (lower = better)
    # Weight: 40% ELO rank, 30% 2026 form, 20% Suzuka factor, 10% grid-to-race model
    elo_rank = current_elo[current_elo['Driver'] == driver].index[0] + 1 if driver in current_elo['Driver'].values else 20
    
    composite = (
        0.40 * elo_rank +
        0.30 * form_2026 +
        0.20 * (form_2026 - suzuka) +  # Suzuka-adjusted form
        0.10 * form_2026  # baseline
    )
    
    # Get team for display
    team = df[(df['Year'] == 2026) & (df['FullName'] == driver)]['TeamName'].iloc[0]
    
    predictions.append({
        'Driver': driver,
        'Team': team,
        'ELO': elo,
        'ELO_Rank': elo_rank,
        'Form_2026': form_2026,
        'Suzuka_Factor': suzuka,
        'Composite': composite
    })

pred_df = pd.DataFrame(predictions).sort_values('Composite').reset_index(drop=True)
pred_df['Predicted_Position'] = range(1, len(pred_df) + 1)

print("=" * 75)
print("  🏁 PREDICTED FINISH ORDER — 2026 JAPANESE GRAND PRIX")
print("=" * 75)
print(f"\n  {'Pos':>3s}  {'Driver':25s}  {'Team':20s}  {'ELO':>7s}  {'Form':>5s}  {'Suzuka':>7s}")
print("  " + "-" * 70)
for _, row in pred_df.iterrows():
    suzuka_str = f"+{row['Suzuka_Factor']:.1f}" if row['Suzuka_Factor'] > 0 else f"{row['Suzuka_Factor']:.1f}" if row['Suzuka_Factor'] != 0 else "  new"
    print(f"  P{row['Predicted_Position']:>2d}  {row['Driver']:25s}  {row['Team']:20s}  {row['ELO']:>7.0f}  P{row['Form_2026']:>4.1f}  {suzuka_str:>7s}")


## 6. Podium ProbabilitiesMonte Carlo simulation: run 10,000 race simulations with randomness to estimate podium probability for each driver.

In [ ]:
# Monte Carlo simulation for podium probabilities
np.random.seed(42)
n_sims = 10000

podium_counts = {driver: 0 for driver in grid_2026}
win_counts = {driver: 0 for driver in grid_2026}
top5_counts = {driver: 0 for driver in grid_2026}

for sim in range(n_sims):
    # Add randomness to composite scores
    sim_results = []
    for _, row in pred_df.iterrows():
        noise = np.random.normal(0, 3)  # Standard deviation of ~3 positions
        sim_results.append((row['Driver'], row['Composite'] + noise))
    
    sim_results.sort(key=lambda x: x[1])
    
    for pos, (driver, _) in enumerate(sim_results):
        if pos == 0:
            win_counts[driver] += 1
        if pos < 3:
            podium_counts[driver] += 1
        if pos < 5:
            top5_counts[driver] += 1

# Build probability table
prob_df = pd.DataFrame([
    {
        'Driver': row['Driver'],
        'Team': row['Team'],
        'Pred_Pos': row['Predicted_Position'],
        'Win%': win_counts[row['Driver']] / n_sims * 100,
        'Podium%': podium_counts[row['Driver']] / n_sims * 100,
        'Top5%': top5_counts[row['Driver']] / n_sims * 100
    }
    for _, row in pred_df.iterrows()
]).sort_values('Win%', ascending=False)

print("=" * 70)
print("  🎲 PODIUM PROBABILITIES (10,000 simulations)")
print("=" * 70)
print(f"\n  {'Driver':25s}  {'Team':20s}  {'Win%':>6s}  {'Podium%':>8s}  {'Top 5%':>7s}")
print("  " + "-" * 68)
for _, row in prob_df.head(10).iterrows():
    win_bar = '█' * int(row['Win%'] / 3)
    print(f"  {row['Driver']:25s}  {row['Team']:20s}  {row['Win%']:>5.1f}%  {row['Podium%']:>7.1f}%  {row['Top5%']:>6.1f}%  {win_bar}")

# Visualize top 10 podium chances
fig, ax = plt.subplots(figsize=(12, 6))
top10 = prob_df.head(10).sort_values('Podium%')
colors = ['#ff006e' if p > 50 else '#3a86ff' if p > 20 else '#8338ec' for p in top10['Podium%']]
bars = ax.barh(top10['Driver'], top10['Podium%'], color=colors)
ax.set_xlabel('Podium Probability (%)', fontsize=12)
ax.set_title('2026 Japanese GP — Podium Probabilities', fontsize=14)
for bar, val in zip(bars, top10['Podium%']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../data/podium_probs.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. F1 Fantasy RecommendationsBased on predicted performance and value, here are the recommended picks for F1 Fantasy this weekend.**F1 Fantasy scoring:**- Race finish: P1=25, P2=18, P3=15, P4=12, P5=10, P6=8, P7=6, P8=4, P9=2, P10=1- Qualifying: Same scale for Q results- Overtakes, fastest lap, and other bonuses- Team constructor scores from both drivers

In [ ]:
# F1 Fantasy expected points
points_map = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

fantasy = []
for _, row in pred_df.iterrows():
    driver = row['Driver']
    pred_pos = row['Predicted_Position']
    
    # Expected race points (weighted by simulation probabilities)
    exp_points = 0
    for pos in range(1, 11):
        # Probability of finishing in this position (from simulation distribution)
        prob = 0
        for sim in range(100):  # Quick estimate
            noise = np.random.normal(0, 3)
            sim_pos = max(1, min(22, int(row['Composite'] + noise)))
            if sim_pos == pos:
                prob += 1
        prob /= 100
        exp_points += prob * points_map.get(pos, 0)
    
    # Overtake potential (grid position vs predicted finish)
    overtake_potential = max(0, row['Form_2026'] - pred_pos)
    
    fantasy.append({
        'Driver': driver,
        'Team': row['Team'],
        'Pred_Finish': pred_pos,
        'Exp_Points': exp_points,
        'Overtake_Bonus': overtake_potential,
        'Total_Value': exp_points + overtake_potential
    })

fantasy_df = pd.DataFrame(fantasy).sort_values('Total_Value', ascending=False)

print("=" * 70)
print("  🏆 F1 FANTASY RECOMMENDATIONS — Japan GP")
print("=" * 70)
print(f"\n  {'Driver':25s}  {'Team':20s}  {'Pred':>5s}  {'Exp Pts':>8s}  {'Value':>6s}")
print("  " + "-" * 68)
for _, row in fantasy_df.head(10).iterrows():
    stars = '⭐' if row['Total_Value'] > 15 else '  '
    print(f"  {row['Driver']:25s}  {row['Team']:20s}  P{row['Pred_Finish']:>3d}  {row['Exp_Points']:>7.1f}  {row['Total_Value']:>5.1f}  {stars}")

print("\n  💡 Top Picks:")
top3 = fantasy_df.head(3)
for _, row in top3.iterrows():
    print(f"     → {row['Driver']} ({row['Team']}) — predicted P{row['Pred_Finish']}, ~{row['Exp_Points']:.0f} expected points")


## 8. Summary & Key Takeaways### Model Notes- **v0.1** — First iteration. Only 2 races of 2026 data, so the model leans heavily on 2025 form and historical Suzuka performance.- ELO ratings will become more accurate as the season progresses.- No weather data or practice session data incorporated yet.- Qualifying predictions are estimated (pre-qualifying). Will be updated after FP1/FP2/FP3.### What to Watch- **Mercedes dominance** — Russell and Antonelli have been 1-2 in both 2026 races. Can they keep it up at Suzuka?- **Verstappen at Suzuka** — Won 3 straight here (2023-2025). Different car, same track knowledge.- **New driver factor** — Several rookies/team-changers (Antonelli, Bearman, Lindblad, Hadjar) have no Suzuka history in F1.### Next Steps- Update predictions after qualifying (Saturday)- Add tire strategy and pit stop modeling- Incorporate practice session pace data- Track weather forecasts for race day